In [1]:
from pathlib import Path
import sys

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

sys.path.insert(0, str(project_root))

from database.db_connection import get_connection, DB_PATH

connection = get_connection()
print("Connected database:", DB_PATH)
connection.close()


Connected database: c:\Users\user\OneDrive\Documents\Desktop\Supplier_Acknowledgement_System\database\supplier_management.db


In [2]:
import pandas as pd

primary_key_columns = {
    "Supplier": "supplier_id",
    "PurchaseOrder": "po_id",
    "POLine": "po_line_id",
    "Acknowledgement": "ack_id",
    "AcknowledgementLine": "ack_line_id",
    "PromiseDate": "promise_id",
    "ChangeRequest": "change_id",
    "Approval": "approval_id",
    "InventoryPlan": "inventory_plan_id",
    "ProductionOrder": "production_order_id",
    "ProductionOrderSupplyAllocation": "allocation_id",
    "Notification": "notification_id",
    "AuditLog": "audit_id",
}

connection = get_connection()
duplicate_results = []

for table, id_column in primary_key_columns.items():
    query = f"""
        SELECT '{table}' AS table_name,
               "{id_column}" AS record_id,
               COUNT(*) AS occurrences
        FROM "{table}"
        GROUP BY "{id_column}"
        HAVING COUNT(*) > 1
    """
    duplicate_results.append(pd.read_sql_query(query, connection))

connection.close()

duplicate_ids = pd.concat(duplicate_results, ignore_index=True)

if duplicate_ids.empty:
    print("PASS: No duplicate IDs found.")
else:
    print("FAIL: Duplicate IDs found:")
    print(duplicate_ids.to_string(index=False))

PASS: No duplicate IDs found.


In [3]:
connection = get_connection()

foreign_key_errors = pd.read_sql_query(
    "PRAGMA foreign_key_check;",
    connection
)

connection.close()

if foreign_key_errors.empty:
    print("PASS: No foreign-key violations found.")
else:
    print("FAIL: Foreign-key violations found:")
    print(foreign_key_errors.to_string(index=False))

PASS: No foreign-key violations found.


In [4]:
status_rules = {
    "Supplier": ("status", {"ACTIVE", "INACTIVE", "BLOCKED"}),
    "PurchaseOrder": (
        "status",
        {"OPEN", "PARTIALLY_ACKNOWLEDGED", "ACKNOWLEDGED", "CLOSED", "CANCELLED"},
    ),
    "Acknowledgement": ("status", {"RECEIVED", "CURRENT", "SUPERSEDED", "INVALID"}),
    "ChangeRequest": (
        "approval_status",
        {"NOT_REQUIRED", "PENDING", "APPROVED", "REJECTED"},
    ),
    "Approval": ("decision", {"PENDING", "APPROVED", "REJECTED"}),
    "InventoryPlan": ("status", {"CURRENT", "SUPERSEDED"}),
    "ProductionOrder": (
        "status",
        {"PLANNED", "IN_PROGRESS", "COMPLETED", "DELAYED", "CANCELLED"},
    ),
    "Notification": (
        "notification_status",
        {"PENDING", "SENT", "READ", "FAILED"},
    ),
}

connection = get_connection()
invalid_statuses = []

for table, (column, allowed_values) in status_rules.items():
    query = f'SELECT DISTINCT "{column}" FROM "{table}"'
    actual_values = pd.read_sql_query(query, connection)[column]

    for value in actual_values:
        if pd.isna(value) or value not in allowed_values:
            invalid_statuses.append(
                {"table": table, "column": column, "invalid_value": value}
            )

connection.close()

invalid_statuses = pd.DataFrame(
    invalid_statuses,
    columns=["table", "column", "invalid_value"],
)

if invalid_statuses.empty:
    print("PASS: All status values are allowed.")
else:
    print("FAIL: Invalid status values found:")
    print(invalid_statuses.to_string(index=False))

PASS: All status values are allowed.


In [5]:
date_check_query = """
    SELECT
        pd.promise_id,
        po.po_id,
        po.order_date,
        pd.promised_date
    FROM PromiseDate AS pd
    JOIN AcknowledgementLine AS al
        ON pd.ack_line_id = al.ack_line_id
    JOIN Acknowledgement AS a
        ON al.ack_id = a.ack_id
       AND al.po_id = a.po_id
    JOIN PurchaseOrder AS po
        ON a.po_id = po.po_id
    WHERE date(pd.promised_date) IS NULL
       OR date(po.order_date) IS NULL
       OR date(pd.promised_date) < date(po.order_date)
"""

connection = get_connection()
delivery_date_issues = pd.read_sql_query(date_check_query, connection)
connection.close()

if delivery_date_issues.empty:
    print("PASS: Delivery dates are consistent with purchase order dates.")
else:
    print("FAIL: Delivery date issues found:")
    print(delivery_date_issues.to_string(index=False))

PASS: Delivery dates are consistent with purchase order dates.


In [6]:
quantity_check_query = """
    SELECT
        al.ack_line_id,
        al.po_id,
        al.po_line_id,
        pol.requested_quantity,
        al.accepted_quantity,
        al.rejected_quantity,
        al.accepted_quantity + al.rejected_quantity AS total_reported_quantity
    FROM AcknowledgementLine AS al
    JOIN POLine AS pol
        ON al.po_id = pol.po_id
       AND al.po_line_id = pol.po_line_id
    WHERE al.accepted_quantity + al.rejected_quantity
          > pol.requested_quantity + 0.000001
"""

connection = get_connection()
quantity_issues = pd.read_sql_query(quantity_check_query, connection)
connection.close()

if quantity_issues.empty:
    print("PASS: Accepted and rejected quantities do not exceed ordered quantities.")
else:
    print("FAIL: Quantity totals exceed the ordered quantities:")
    print(quantity_issues.to_string(index=False))

PASS: Accepted and rejected quantities do not exceed ordered quantities.


In [7]:
negative_value_query = """
    SELECT 'POLine' AS table_name, po_line_id AS record_id,
           'requested_quantity' AS field_name, requested_quantity AS invalid_value
    FROM POLine
    WHERE requested_quantity <= 0

    UNION ALL

    SELECT 'POLine', po_line_id, 'unit_price', unit_price
    FROM POLine
    WHERE unit_price < 0

    UNION ALL

    SELECT 'AcknowledgementLine', ack_line_id, 'accepted_quantity', accepted_quantity
    FROM AcknowledgementLine
    WHERE accepted_quantity < 0

    UNION ALL

    SELECT 'AcknowledgementLine', ack_line_id, 'rejected_quantity', rejected_quantity
    FROM AcknowledgementLine
    WHERE rejected_quantity < 0

    UNION ALL

    SELECT 'AcknowledgementLine', ack_line_id, 'proposed_unit_price', proposed_unit_price
    FROM AcknowledgementLine
    WHERE proposed_unit_price < 0

    UNION ALL

    SELECT 'InventoryPlan', inventory_plan_id, 'projected_quantity', projected_quantity
    FROM InventoryPlan
    WHERE projected_quantity < 0

    UNION ALL

    SELECT 'ProductionOrder', production_order_id, 'required_quantity', required_quantity
    FROM ProductionOrder
    WHERE required_quantity <= 0

    UNION ALL

    SELECT 'ProductionOrderSupplyAllocation', allocation_id, 'allocated_quantity',
           allocated_quantity
    FROM ProductionOrderSupplyAllocation
    WHERE allocated_quantity <= 0

    UNION ALL

    SELECT 'ChangeRequest', change_id, 'old_value (' || change_type || ')', old_value
    FROM ChangeRequest
    WHERE change_type IN ('QUANTITY', 'PRICE')
      AND CAST(old_value AS REAL) < 0

    UNION ALL

    SELECT 'ChangeRequest', change_id, 'new_value (' || change_type || ')', new_value
    FROM ChangeRequest
    WHERE change_type IN ('QUANTITY', 'PRICE')
      AND CAST(new_value AS REAL) < 0
"""

connection = get_connection()
negative_value_issues = pd.read_sql_query(negative_value_query, connection)
connection.close()

if negative_value_issues.empty:
    print("PASS: No invalid negative or zero quantity/price values found.")
else:
    print("FAIL: Invalid quantity or price values found:")
    print(negative_value_issues.to_string(index=False))

PASS: No invalid negative or zero quantity/price values found.


In [8]:
revision_sequence_query = """
    SELECT
        po_id,
        COUNT(*) AS revision_count,
        MIN(revision_no) AS first_revision,
        MAX(revision_no) AS last_revision
    FROM Acknowledgement
    GROUP BY po_id
    HAVING MIN(revision_no) <> 1
        OR MAX(revision_no) <> COUNT(*)
"""

multiple_current_query = """
    SELECT
        po_id,
        COUNT(*) AS current_revision_count
    FROM Acknowledgement
    WHERE status = 'CURRENT'
    GROUP BY po_id
    HAVING COUNT(*) > 1
"""

connection = get_connection()
revision_sequence_issues = pd.read_sql_query(
    revision_sequence_query, connection
)
multiple_current_issues = pd.read_sql_query(
    multiple_current_query, connection
)
connection.close()

if revision_sequence_issues.empty and multiple_current_issues.empty:
    print("PASS: Acknowledgement revisions are consistent.")
else:
    print("FAIL: Acknowledgement revision issues found.")

    if not revision_sequence_issues.empty:
        print("\nRevision sequence issues:")
        print(revision_sequence_issues.to_string(index=False))

    if not multiple_current_issues.empty:
        print("\nMultiple CURRENT acknowledgements for a PO:")
        print(multiple_current_issues.to_string(index=False))

PASS: Acknowledgement revisions are consistent.


In [9]:
inventory_approval_query = """
    SELECT
        ip.inventory_plan_id,
        ip.po_line_id,
        ip.source_change_id,
        cr.approval_status
    FROM InventoryPlan AS ip
    JOIN ChangeRequest AS cr
        ON ip.source_change_id = cr.change_id
    WHERE cr.approval_status <> 'APPROVED'
"""

approved_change_query = """
    SELECT
        cr.change_id,
        cr.approval_status,
        COUNT(a.approval_id) AS approval_count,
        SUM(
            CASE WHEN a.decision <> 'APPROVED' THEN 1 ELSE 0 END
        ) AS not_approved_count
    FROM ChangeRequest AS cr
    LEFT JOIN Approval AS a
        ON cr.change_id = a.change_id
    WHERE cr.approval_status = 'APPROVED'
    GROUP BY cr.change_id, cr.approval_status
    HAVING COUNT(a.approval_id) = 0
        OR SUM(
            CASE WHEN a.decision <> 'APPROVED' THEN 1 ELSE 0 END
        ) > 0
"""

connection = get_connection()
unapproved_inventory_updates = pd.read_sql_query(
    inventory_approval_query, connection
)
incomplete_approved_changes = pd.read_sql_query(
    approved_change_query, connection
)
connection.close()

if unapproved_inventory_updates.empty and incomplete_approved_changes.empty:
    print("PASS: Inventory updates and approvals are consistent.")
else:
    print("FAIL: Approval or inventory issues found.")

    if not unapproved_inventory_updates.empty:
        print("\nInventory plans linked to non-approved changes:")
        print(unapproved_inventory_updates.to_string(index=False))

    if not incomplete_approved_changes.empty:
        print("\nChanges marked APPROVED without completed approvals:")
        print(incomplete_approved_changes.to_string(index=False))

PASS: Inventory updates and approvals are consistent.


In [10]:
audit_duplicate_query = """
    SELECT
        user,
        action,
        description,
        timestamp,
        COUNT(*) AS occurrences
    FROM AuditLog
    GROUP BY user, action, description, timestamp
    HAVING COUNT(*) > 1
"""

notification_duplicate_query = """
    SELECT
        po_id,
        change_id,
        production_order_id,
        recipient_role,
        message,
        created_at,
        COUNT(*) AS occurrences
    FROM Notification
    GROUP BY
        po_id,
        change_id,
        production_order_id,
        recipient_role,
        message,
        created_at
    HAVING COUNT(*) > 1
"""

connection = get_connection()
duplicate_audit_events = pd.read_sql_query(
    audit_duplicate_query, connection
)
duplicate_notifications = pd.read_sql_query(
    notification_duplicate_query, connection
)
connection.close()

if duplicate_audit_events.empty and duplicate_notifications.empty:
    print("PASS: No duplicate audit events or notifications found.")
else:
    print("FAIL: Duplicate events or notifications found.")

    if not duplicate_audit_events.empty:
        print("\nDuplicate audit events:")
        print(duplicate_audit_events.to_string(index=False))

    if not duplicate_notifications.empty:
        print("\nDuplicate notifications:")
        print(duplicate_notifications.to_string(index=False))

PASS: No duplicate audit events or notifications found.


In [11]:
ground_truth_path = (
    project_root / "data" / "ground_truth" / "ground_truth.csv"
)
ground_truth_df = pd.read_csv(ground_truth_path)

ground_truth_ids = set(
    ground_truth_df["anomaly_id"].dropna().astype(str)
)
production_ids = set()

connection = get_connection()

for table, id_column in primary_key_columns.items():
    query = f'SELECT "{id_column}" FROM "{table}"'
    table_ids = pd.read_sql_query(query, connection)[id_column]
    production_ids.update(table_ids.dropna().astype(str))

connection.close()

overlapping_ids = ground_truth_ids.intersection(production_ids)

if not overlapping_ids:
    print("PASS: Ground-truth IDs are separate from production record IDs.")
else:
    print("FAIL: Ground-truth IDs found in production tables:")
    print(sorted(overlapping_ids))

PASS: Ground-truth IDs are separate from production record IDs.


In [12]:
validation_summary = pd.DataFrame([
    {
        "check_id": "DV01",
        "check": "Duplicate primary-key IDs",
        "result": "PASS" if duplicate_ids.empty else "FAIL",
        "issue_count": len(duplicate_ids),
    },
    {
        "check_id": "DV02",
        "check": "Foreign-key relationships",
        "result": "PASS" if foreign_key_errors.empty else "FAIL",
        "issue_count": len(foreign_key_errors),
    },
    {
        "check_id": "DV03",
        "check": "Allowed status values",
        "result": "PASS" if invalid_statuses.empty else "FAIL",
        "issue_count": len(invalid_statuses),
    },
    {
        "check_id": "DV04",
        "check": "Delivery-date consistency",
        "result": "PASS" if delivery_date_issues.empty else "FAIL",
        "issue_count": len(delivery_date_issues),
    },
    {
        "check_id": "DV05",
        "check": "Accepted and rejected quantities",
        "result": "PASS" if quantity_issues.empty else "FAIL",
        "issue_count": len(quantity_issues),
    },
    {
        "check_id": "DV06",
        "check": "Quantity and price ranges",
        "result": "PASS" if negative_value_issues.empty else "FAIL",
        "issue_count": len(negative_value_issues),
    },
    {
        "check_id": "DV07",
        "check": "Acknowledgement revision consistency",
        "result": (
            "PASS"
            if revision_sequence_issues.empty and multiple_current_issues.empty
            else "FAIL"
        ),
        "issue_count": (
            len(revision_sequence_issues) + len(multiple_current_issues)
        ),
    },
    {
        "check_id": "DV08",
        "check": "Approval and inventory consistency",
        "result": (
            "PASS"
            if unapproved_inventory_updates.empty
            and incomplete_approved_changes.empty
            else "FAIL"
        ),
        "issue_count": (
            len(unapproved_inventory_updates)
            + len(incomplete_approved_changes)
        ),
    },
    {
        "check_id": "DV09",
        "check": "Duplicate audit events and notifications",
        "result": (
            "PASS"
            if duplicate_audit_events.empty and duplicate_notifications.empty
            else "FAIL"
        ),
        "issue_count": len(duplicate_audit_events) + len(duplicate_notifications),
    },
    {
        "check_id": "DV10",
        "check": "Ground-truth ID separation",
        "result": "PASS" if not overlapping_ids else "FAIL",
        "issue_count": len(overlapping_ids),
    },
])

summary_path = project_root / "reports" / "validation_summary.csv"
validation_summary.to_csv(summary_path, index=False)

print(validation_summary.to_string(index=False))
print("\nSaved validation summary to:", summary_path)

check_id                                    check result  issue_count
    DV01                Duplicate primary-key IDs   PASS            0
    DV02                Foreign-key relationships   PASS            0
    DV03                    Allowed status values   PASS            0
    DV04                Delivery-date consistency   PASS            0
    DV05         Accepted and rejected quantities   PASS            0
    DV06                Quantity and price ranges   PASS            0
    DV07     Acknowledgement revision consistency   PASS            0
    DV08       Approval and inventory consistency   PASS            0
    DV09 Duplicate audit events and notifications   PASS            0
    DV10               Ground-truth ID separation   PASS            0

Saved validation summary to: c:\Users\user\OneDrive\Documents\Desktop\Supplier_Acknowledgement_System\reports\validation_summary.csv


In [13]:
import platform
import sqlite3
import sys

passed_count = (validation_summary["result"] == "PASS").sum()
failed_count = (validation_summary["result"] == "FAIL").sum()
database_relative_path = Path(DB_PATH).relative_to(project_root).as_posix()

report_text = f"""# Reproducibility Report

## Environment
- Operating system: {platform.platform()}
- Python: {sys.version.split()[0]}
- Pandas: {pd.__version__}
- SQLite: {sqlite3.sqlite_version}
- Database: `{database_relative_path}`
- Notebook: `notebooks/02_data_validation.ipynb`

## How to reproduce
1. Open the project folder in VS Code.
2. Open `notebooks/02_data_validation.ipynb`.
3. Select the project's Python virtual environment kernel.
4. Ensure the database and `data/ground_truth/ground_truth.csv` are present.
5. Run all notebook cells from top to bottom.
6. Review the results in `reports/validation_summary.csv`.

The validation queries read the database. The notebook writes the validation summary report.

## Results
- Validation checks: {len(validation_summary)}
- Passed: {passed_count}
- Failed: {failed_count}
- Summary file: `reports/validation_summary.csv`

The ground-truth check compares scenario IDs with production primary-key IDs; it does not compare every database text value.
"""

report_path = project_root / "reports" / "reproducibility_report.md"
report_path.write_text(report_text, encoding="utf-8")

print("Saved reproducibility report to:", report_path)
print(f"Passed: {passed_count}; Failed: {failed_count}")

Saved reproducibility report to: c:\Users\user\OneDrive\Documents\Desktop\Supplier_Acknowledgement_System\reports\reproducibility_report.md
Passed: 10; Failed: 0
